In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Tree Ensembles Lab

Baseline chỉ phụ thuộc scikit-learn. XGBoost/LightGBM/CatBoost là optional nếu đúng environment đã có.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score
n=600 if FAST_MODE else 5000
X,y=make_classification(n_samples=n,n_features=20,n_informative=10,random_state=42)
Xtr,Xva,ytr,yva=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
models={"random_forest":RandomForestClassifier(n_estimators=30 if FAST_MODE else 200,random_state=42,n_jobs=-1),"hist_gradient_boosting":HistGradientBoostingClassifier(max_iter=30 if FAST_MODE else 150,random_state=42)}
for name,model in models.items():
    start=time.perf_counter(); model.fit(Xtr,ytr); score=accuracy_score(yva,model.predict(Xva)); elapsed=time.perf_counter()-start
    print(name, 'accuracy', round(score,3))
    assert score>.75

In [ ]:
import importlib.util
optional={name:bool(importlib.util.find_spec(name)) for name in ("xgboost","lightgbm","catboost")}
print("optional libraries already available:",optional)
# Không tự cài package. Chỉ benchmark các thư viện optional khi competition profile xác nhận có sẵn.